# Semantic Symptom Mapping Using Vector Embeddings

This notebook demonstrates the semantic-mapping step of the NeSy-X execution pipeline: extracted symptoms are embedded with **intfloat/multilingual-e5-large** and matched against the ontological symptom embeddings stored in Neo4j using cosine similarity. The matched, mapped symptoms are then saved so the next notebook (`03-inference-and-scoring.ipynb`) can pick them up as its input.

## Assumption

This experiment assumes the existence of a preceding NLP layer that has already extracted relevant symptoms from the user's natural language input. Therefore, **the process of extracting symptoms from raw text is not considered here** — see `01-nlp-llm.ipynb`. The goal here is to evaluate semantic mapping of extracted symptoms to ontological concepts using vector embeddings, in isolation from the downstream symbolic reasoning step.

In [ ]:
# [1] No absent symptoms
# symptom_input = {"present": ["headache", "fever", "stiff neck", "vomiting", "seizure"], "absent": []}

# [2] Should  return 'West Nile encephalitis', 	Japanese encephalitis eliminated
#symptom_input = {"present": ["headache", "high fever", "coma", "confusion", "stiff neck"], "absent": ["spastic paralysis"]}

# [3] Should return 'hepatitis E', 'hepatitis' D eliminated
# symptom_input = {"present": ["jaundice", "nausea", "fatigue", "dark urine", "acholic stool"], "absent": ["confusion", "drowsiness"]} 

# [4] Should return 'Ebola virus disease', 'Marburg hemorrhagic fever' eliminated
#symptom_input = {"present": ["fever", "headache", "vomiting", "bleeding", "pharynx inflammation"], "absent": ["maculopapular rash", "chills"]}

symptom_input = {"present": ["abdominal pain", "rash", "fever", "bloodshot eye", "chills", "shock"], "absent": ["back pain"]}


# Step 1: Vector Embedding

Each extracted symptom is mapped into a continuous vector space using the pre-trained **intfloat/multilingual-e5-large** model. This phase enables semantic comparison between symptoms provided by the user and symptoms defined in the ontology.

Embeddings for ontological symptoms are generated during a preprocessing phase and persisted in the graph database, while embeddings for input symptoms are computed at inference time for each user query.

In [ ]:
from sentence_transformers import SentenceTransformer

#model = SentenceTransformer('all-MiniLM-L6-v2')
#model = SentenceTransformer('NeuML/pubmedbert-base-embeddings')
model = SentenceTransformer('intfloat/multilingual-e5-large')

all_input_symptoms = symptom_input["present"] + symptom_input["absent"]
query_embeddings = model.encode(all_input_symptoms)

## Loading Ontological Symbols and Vector Embeddings

In [ ]:
import os
from neo4j import GraphDatabase
from dotenv import load_dotenv

load_dotenv(override=True)

neo4j_url      = os.getenv("NEO4J_URL")
neo4j_username = os.getenv("NEO4J_USERNAME")
neo4j_password = os.getenv("NEO4J_PASSWORD")

driver = GraphDatabase.driver(neo4j_url, auth=(neo4j_username, neo4j_password))

with driver.session() as session:
    result = session.run("""
        MATCH (s:Symptom)
        RETURN s.n4sch__label[0] AS label, s.embedding AS embedding
    """)
    ontology_symptoms = [(r["label"], r["embedding"]) for r in result]

print(f"Loaded {len(ontology_symptoms)} ontological symptoms.")

# Step 2: Semantic Symptom Mapping

Cosine similarity is used to compare the vector embeddings of input symptoms against ontological symptoms stored in the Neo4j graph database.
For each input symptom, the ontological term with the highest similarity score is identified and used as its semantic equivalent in the downstream inference process.

## Symptom Matching via Cosine Similarity

Each input symptom embedding is compared against all ontological symptom embeddings using cosine similarity. The ontological term with the highest similarity score is selected as the semantic match. A confidence threshold of **0.9** is applied — symptoms that fall below this threshold are excluded from the inference step to avoid noisy or ambiguous mappings.

Input symptoms are split into two lists based on their kind:
- **Present symptoms** — mapped ontological terms that will be used to find matching diseases,
- **Absent symptoms** — mapped ontological terms that will be used to exclude diseases.

In [ ]:
import json
from pathlib import Path

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

onto_labels  = [o[0] for o in ontology_symptoms]
onto_vectors = np.array([o[1] for o in ontology_symptoms])

CONFIDENCE_THRESHOLD = 0.9

matched = []
for symptom, q_emb in zip(all_input_symptoms, query_embeddings):
    sims     = cosine_similarity([q_emb], onto_vectors)[0]
    best_idx = sims.argmax()
    matched.append({
        "input_symptom":  symptom,
        "mapped_symptom": onto_labels[best_idx],
        "confidence":     float(sims[best_idx]),
        "kind":           "present" if symptom in symptom_input["present"] else "absent"
    })

print(f"{'USER INPUT':<25} | {'KIND':<8} | {'ONTOLOGICAL TERM':<30} | {'SIMILARITY'}")
print("-" * 85)
for m in matched:
    flag = "" if m["confidence"] > CONFIDENCE_THRESHOLD else " -- UNDER THRESHOLD"
    print(f"{m['input_symptom']:<25} | {m['kind']:<8} | {m['mapped_symptom']:<30} | {m['confidence']:.4f}{flag}")

present_symptoms = [
    m["mapped_symptom"] for m in matched
    if m["kind"] == "present" and m["confidence"] > CONFIDENCE_THRESHOLD
]
absent_symptoms = [
    m["mapped_symptom"] for m in matched
    if m["kind"] == "absent" and m["confidence"] > CONFIDENCE_THRESHOLD
]

print(f"\nPresent: {present_symptoms}")
print(f"Absent:  {absent_symptoms}")

## Saving the Mapped Symptoms

`present_symptoms` and `absent_symptoms` are written to a small JSON file so `03-inference-and-scoring.ipynb` can load them as its input without needing to re-run the embedding model. This mirrors how the preparation notebooks hand off state through the Neo4j graph rather than in-memory variables.

In [ ]:
RESULTS_DIR = Path("results") / "semantic-mapping"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

output_path = RESULTS_DIR / "matched_symptoms.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(
        {"present_symptoms": present_symptoms, "absent_symptoms": absent_symptoms},
        f,
        indent=2,
        ensure_ascii=False,
    )

print(f"Mapped symptoms saved to '{output_path}'.")
driver.close()

## Embedding Model Evaluation

The results and methodology of the embedding model evaluation are described in detail in **embedding-test**.

In [ ]:
from pathlib import Path
import json

BASE_DIR = Path.cwd()

test_cases_path = BASE_DIR / "tests" / "embeddings-test" / "symptom-full-test.json"

with open(test_cases_path, "r", encoding="utf-8") as f:
    test_cases = json.load(f)

print(f"{len(test_cases)} test cases loaded.")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import numpy as np
import json
import os

# model = SentenceTransformer('all-MiniLM-L6-v2')
# model = SentenceTransformer('NeuML/pubmedbert-base-embeddings')
model = SentenceTransformer('intfloat/multilingual-e5-large')

all_embeddings = []
all_metadata = []
matched = {}

for case in test_cases:
    symptoms = case.get("extracted_symptoms", [])
    case_id = case.get("id", "unknown")

    if not symptoms:
        continue

    if case_id not in matched:
        matched[case_id] = []

    for symptom in symptoms:
        embedding = model.encode(symptom.strip(), normalize_embeddings=True)
        onto_labels = [o[0] for o in ontology_symptoms]
        onto_vectors = np.array([o[1] for o in ontology_symptoms])

        sims = cosine_similarity([embedding], onto_vectors)[0]
        best_idx = sims.argmax()

        matched[case_id].append({
            "input_symptom": symptom,
            "mapped_symptom": onto_labels[best_idx],
            "confidence": float(sims[best_idx])
        })

output_dir = "tests/embeddings-test/results"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, "embedding-test-3.json")

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(matched, f, indent=2, ensure_ascii=False)

print(f"Results written at: {output_path}")
print(f"Total cases: {len(matched)}")

In [ ]:
def evaluate_embedding_layer(matched_results):
    usable_threshold = 0.90

    def empty_stats():
        return {
            "total": 0,
            "exact": 0,
            "usable": 0,
            "fail": 0,
            "conf_sum": 0.0,
            "fail_cases": [],
            "non_exact_usable": []
        }

    categories = {
        "EXACT":  empty_stats(),
        "SYN":    empty_stats(),
        "LLM":   empty_stats(),
        "SUM": empty_stats(),
    }

    for case_id, mappings in matched_results.items():
        if not mappings:
            continue

        best = mappings[0]
        conf      = best["confidence"]
        input_sym = best["input_symptom"]
        mapped_sym = best["mapped_symptom"]
        is_exact  = (input_sym.strip().lower() == mapped_sym.strip().lower())

        if case_id.startswith("EXACT"):
            cat = "EXACT"
        elif case_id.startswith("SYN"):
            cat = "SYN"
        elif case_id.startswith("LLM"):
            cat = "LLM"
        else:
            cat = "SUM"

        for key in [cat, "SUM"]:
            s = categories[key]
            s["total"]    += 1
            s["conf_sum"] += conf

            if is_exact:
                s["exact"] += 1

            if conf >= usable_threshold:
                s["usable"] += 1
                if not is_exact:
                    s["non_exact_usable"].append((case_id, input_sym, mapped_sym, conf))
            else:
                s["fail"] += 1
                s["fail_cases"].append((case_id, input_sym, mapped_sym, conf))

    def print_category(name, s):
        total = s["total"]
        if total == 0:
            print(f"\n[{name}] — No data available.")
            return

        exact_rate   = (s["exact"]  / total) * 100
        usable_rate  = (s["usable"] / total) * 100
        bad_rate     = (s["fail"]   / total) * 100
        avg_conf     = s["conf_sum"] / total

        is_summary = (name == "SUM")
        sep = "═" * 60 if is_summary else "─" * 60

        print(f"\n{sep}")
        if is_summary:
            print(f"{'  SUMMARY — ALL CATEGORIES  ':^60}")
        else:
            desc = {"EXACT": "180 terms from ontology", "SYN": "70 synonyms", "LLM": "llm cases"}.get(name, "")
            print(f"{'  CATEGORY: ' + name + ' (' + str(total) + ' tests' + ((' — ' + desc) if desc else '') + ')  ':^60}")
        print(sep)

        print(f"  {'Metric':<32} | {'Value':>9} | Status")
        print(f"  {'─'*32}─+─{'─'*9}─+─{'─'*6}")

        print(f"  {'Exact Match Rate':<32} | {exact_rate:>8.1f}%")
        print(f"  {'Usable Match Rate  (conf ≥ 0.90)':<32} | {usable_rate:>8.1f}%")
        print(f"  {'Average Confidence':<32} | {avg_conf:>9.4f}")
        print(f"  {'Bad Match Rate     (conf < 0.90)':<32} | {bad_rate:>8.1f}%")
        print(f"  {'─'*55}")
        print(f"  Exact: {s['exact']}  |  Usable (non-exact): {len(s['non_exact_usable'])}  |  Fail: {s['fail']}  |  Total: {total}")

    for cat in ["EXACT", "SYN", "LLM"]:
        print_category(cat, categories[cat])

    print_category("SUM", categories["SUM"])

    return categories

In [ ]:
results = evaluate_embedding_layer(matched)